In [ ]:
import utils
import ot
import matplotlib.pyplot as plt

In [ ]:
points = utils.sampled_verts_from_random_action()

In [ ]:
utils.plot_arr(points)

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

class Skeleton:
    def __init__(self):
        self.vertices = []  # list of (x, y, z)
        self.edges = []     # list of (i, j)

    def add_vertex(self, v):
        self.vertices.append(v)
        return len(self.vertices) - 1

    def add_edge(self, i, j):
        self.edges.append((i, j))

    def as_arrays(self):
        return np.array(self.vertices), self.edges


import numpy as np


def construct_skeleton(point_cloud):
    """
    point_cloud: (N, 3) numpy array
    returns:
        vertices: (16, 3) numpy array (ordered landmarks)
        edges: list of (i, j)
    """

    pc = point_cloud
    x, y, z = pc[:, 0], pc[:, 1], pc[:, 2]

    z_min, z_max = z.min(), z.max()
    height = z_max - z_min

    def z_slice(z_center, thickness=0.04):
        return pc[np.abs(z - z_center) < thickness * height]

    def centroid(pts):
        return pts.mean(axis=0)

    def left_right(pts):
        left = pts[np.argmin(pts[:, 0])]
        right = pts[np.argmax(pts[:, 0])]
        return left, right

    # --- vertical reference levels ---
    levels = {
        "head": 0.95,
        "upper_torso": 0.85,
        "lower_torso": 0.60,
        "pelvis": 0.50,
        "thigh": 0.35,
        "shin": 0.20,
        "foot": 0.05,
        "upper_arm": 0.70,
        "forearm": 0.60,
        "hand": 0.50,
    }

    verts = {}

    # --- torso / head ---
    verts["head"] = centroid(z_slice(z_min + levels["head"] * height))
    verts["upper_torso"] = centroid(z_slice(z_min + levels["upper_torso"] * height))
    verts["lower_torso"] = centroid(z_slice(z_min + levels["lower_torso"] * height))
    verts["pelvis"] = centroid(z_slice(z_min + levels["pelvis"] * height))

    # --- legs ---
    thigh_pts = z_slice(z_min + levels["thigh"] * height)
    left_thigh, right_thigh = left_right(thigh_pts)

    shin_pts = z_slice(z_min + levels["shin"] * height)
    left_shin, right_shin = left_right(shin_pts)

    foot_pts = z_slice(z_min + levels["foot"] * height)
    left_foot, right_foot = left_right(foot_pts)

    verts["left_thigh"] = left_thigh
    verts["right_thigh"] = right_thigh
    verts["left_shin"] = left_shin
    verts["right_shin"] = right_shin
    verts["left_foot"] = left_foot
    verts["right_foot"] = right_foot

    # --- arms ---
    upper_arm_pts = z_slice(z_min + levels["upper_arm"] * height)
    left_upper_arm, right_upper_arm = left_right(upper_arm_pts)

    forearm_pts = z_slice(z_min + levels["forearm"] * height)
    left_forearm, right_forearm = left_right(forearm_pts)

    hand_pts = z_slice(z_min + levels["hand"] * height)
    left_hand, right_hand = left_right(hand_pts)

    verts["left_upper_arm"] = left_upper_arm
    verts["right_upper_arm"] = right_upper_arm
    verts["left_forearm"] = left_forearm
    verts["right_forearm"] = right_forearm
    verts["left_hand"] = left_hand
    verts["right_hand"] = right_hand

    # --- assemble vertex array in required order ---
    landmark_order = [
        "head",
        "left_foot",
        "left_forearm",
        "left_hand",
        "left_shin",
        "left_thigh",
        "left_upper_arm",
        "lower_torso",
        "pelvis",
        "right_foot",
        "right_forearm",
        "right_hand",
        "right_shin",
        "right_thigh",
        "right_upper_arm",
        "upper_torso",
    ]

    vertices = np.array([verts[name] for name in landmark_order])

    # --- edges (bone structure) ---
    edges = [
        (0, 15),   # head → upper_torso
        (15, 7),   # upper_torso → lower_torso
        (7, 8),    # lower_torso → pelvis

        (8, 5),    # pelvis → left_thigh
        (5, 4),    # left_thigh → left_shin
        (4, 1),    # left_shin → left_foot

        (8, 13),   # pelvis → right_thigh
        (13, 12),  # right_thigh → right_shin
        (12, 9),   # right_shin → right_foot

        (15, 6),   # upper_torso → left_upper_arm
        (6, 2),    # left_upper_arm → left_forearm
        (2, 3),    # left_forearm → left_hand

        (15, 14),  # upper_torso → right_upper_arm
        (14, 10),  # right_upper_arm → right_forearm
        (10, 11),  # right_forearm → right_hand
    ]

    return vertices, edges



In [ ]:
skeleton = construct_skeleton(points)

In [ ]:
import numpy as np
import plotly.express as px
import pandas as pd


def plot_pointcloud_and_skeleton(point_cloud, vertices, edges):
    """
    point_cloud: (N, 3) numpy array
    vertices: (M, 3) numpy array
    edges: list of (i, j)
    """

    # --- Point cloud ---
    pc_df = pd.DataFrame(point_cloud, columns=["x", "y", "z"])
    pc_df["type"] = "point_cloud"

    # --- Skeleton vertices ---
    v_df = pd.DataFrame(vertices, columns=["x", "y", "z"])
    v_df["type"] = "skeleton_vertex"

    # --- Skeleton edges (as line segments) ---
    edge_x = []
    edge_y = []
    edge_z = []

    for i, j in edges:
        edge_x.extend([vertices[i, 0], vertices[j, 0], None])
        edge_y.extend([vertices[i, 1], vertices[j, 1], None])
        edge_z.extend([vertices[i, 2], vertices[j, 2], None])

    edge_df = pd.DataFrame({
        "x": edge_x,
        "y": edge_y,
        "z": edge_z
    })

    # --- Plot point cloud ---
    fig = px.scatter_3d(
        pc_df,
        x="x",
        y="y",
        z="z",
        opacity=0.15,
        color_discrete_sequence=["lightgray"]
    )

    # --- Add skeleton vertices ---
    fig.add_scatter3d(
        x=v_df["x"],
        y=v_df["y"],
        z=v_df["z"],
        mode="markers",
        marker=dict(size=6, color="red"),
        name="Skeleton vertices"
    )

    # --- Add skeleton edges ---
    fig.add_scatter3d(
        x=edge_df["x"],
        y=edge_df["y"],
        z=edge_df["z"],
        mode="lines",
        line=dict(width=6, color="black"),
        name="Skeleton edges"
    )

    # --- Layout tweaks ---
    fig.update_layout(
        scene=dict(
            xaxis_title="X",
            yaxis_title="Y",
            zaxis_title="Z",
            aspectmode="data"
        ),
        legend=dict(itemsizing="constant")
    )

    fig.update_layout(scene = dict(aspectmode = "data"))
    fig.update_layout(
        width=600,
        height=1000,   # taller than wide works better for humans
    )
    fig.update_layout(
        scene_camera=dict(
            eye=dict(x=2.5, y=2.5, z=2.5)
        )
    )
    return fig


In [ ]:
fig = plot_pointcloud_and_skeleton(points, skeleton[0], skeleton[1])
fig.show()

In [ ]:
import numpy as np
import heapq

def nearest_skeleton_vertices(points, vertices):
    """
    Returns:
        nearest_idx: (N,) index of nearest skeleton vertex
        nearest_dist: (N,) distance to nearest skeleton vertex
    """
    diff = points[:, None, :] - vertices[None, :, :]
    dists = np.linalg.norm(diff, axis=2)

    nearest_idx = np.argmin(dists, axis=1)
    nearest_dist = dists[np.arange(len(points)), nearest_idx]

    return nearest_idx, nearest_dist

def skeleton_graph_distances(vertices, edges):
    """
    Returns:
        D: (M, M) matrix of shortest-path distances along skeleton
    """
    M = len(vertices)

    # adjacency list
    adj = [[] for _ in range(M)]
    for i, j in edges:
        w = np.linalg.norm(vertices[i] - vertices[j])
        adj[i].append((j, w))
        adj[j].append((i, w))

    def dijkstra(src):
        dist = np.full(M, np.inf)
        dist[src] = 0.0
        pq = [(0.0, src)]

        while pq:
            d, u = heapq.heappop(pq)
            if d > dist[u]:
                continue
            for v, w in adj[u]:
                nd = d + w
                if nd < dist[v]:
                    dist[v] = nd
                    heapq.heappush(pq, (nd, v))
        return dist

    D = np.zeros((M, M))
    for i in range(M):
        D[i] = dijkstra(i)

    return D

def point_distance_matrix(points, vertices, edges):
    """
    Returns:
        D_points: (N, N) distance matrix
    """

    N = len(points)

    # nearest skeleton vertices
    nearest_idx, nearest_dist = nearest_skeleton_vertices(points, vertices)

    # skeleton graph distances
    D_skel = skeleton_graph_distances(vertices, edges)

    # assemble full matrix
    D = np.zeros((N, N))

    for i in range(N):
        si = nearest_idx[i]
        di = nearest_dist[i]

        for j in range(N):
            sj = nearest_idx[j]
            dj = nearest_dist[j]

            D[i, j] = di + D_skel[si, sj] + dj

    return D


In [ ]:
vertices, edges = construct_skeleton(points)
D = point_distance_matrix(points, vertices, edges)

print(D.shape)  # (N, N)

In [ ]:
def distances_from_point_0(points, vertices, edges):
    nearest_idx, nearest_dist = nearest_skeleton_vertices(points, vertices)
    D_skel = skeleton_graph_distances(vertices, edges)

    s0 = nearest_idx[0]
    d0 = nearest_dist[0]

    dists = np.zeros(len(points))
    for i in range(len(points)):
        si = nearest_idx[i]
        di = nearest_dist[i]
        dists[i] = d0 + D_skel[s0, si] + di

    return dists


In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np


In [ ]:
def plot_colored_pointcloud_with_skeleton(points, vertices, edges):
    # distances from point 0
    dists = distances_from_point_0(points, vertices, edges)

    # --- point cloud dataframe ---
    pc_df = pd.DataFrame(points, columns=["x", "y", "z"])
    pc_df["distance"] = dists

    # --- skeleton edges ---
    edge_x, edge_y, edge_z = [], [], []
    for i, j in edges:
        edge_x += [vertices[i, 0], vertices[j, 0], None]
        edge_y += [vertices[i, 1], vertices[j, 1], None]
        edge_z += [vertices[i, 2], vertices[j, 2], None]

    # --- plot point cloud ---
    fig = px.scatter_3d(
        pc_df,
        x="x",
        y="y",
        z="z",
        color="distance",
        color_continuous_scale="viridis",
        opacity=0.7,
        title="Skeleton-induced distance from point 0"
    )

    # --- skeleton vertices ---
    fig.add_scatter3d(
        x=vertices[:, 0],
        y=vertices[:, 1],
        z=vertices[:, 2],
        mode="markers",
        marker=dict(size=6, color="red"),
        name="Skeleton vertices"
    )

    # --- skeleton edges ---
    fig.add_scatter3d(
        x=edge_x,
        y=edge_y,
        z=edge_z,
        mode="lines",
        line=dict(width=6, color="black"),
        name="Skeleton edges"
    )

    fig.update_layout(
        scene=dict(aspectmode="data"),
        legend=dict(itemsizing="constant")
    )

    fig.show()


In [ ]:
vertices, edges = construct_skeleton(points)
plot_colored_pointcloud_with_skeleton(points, vertices, edges)


In [ ]:
def compute_disambiguated_axes(feature_index, verts, R, return_diffs = True, return_mask = True):
    # convert to numpy
    V = verts
    p0 = V[feature_index]

    # distances
    diffs = V - p0
    dists = np.linalg.norm(diffs, axis=1)

    # only take points within R radius
    mask = dists < R
    diffs = diffs[mask]
    dists = dists[mask]

    # distance-based weights
    w = (R - dists)
    w_sum = np.sum(w)

    if w_sum == 0:
        raise ValueError("No neighbors inside radius")

    # weighted covariance matrix
    M = (diffs.T * w) @ diffs / w_sum

    # eigendecomp to find approximate normal, as well as other axes
    eigvals, eigvecs = np.linalg.eigh(M)

    # smallest eigvec is the normal (z), biggest is x
    normal = eigvecs[:, 0]
    x_vec = eigvecs[:, 2]

    # axis disambiguation
    S_plus = (diffs @ normal >= 0).sum()
    S_minus = diffs.shape[0] - S_plus
    
    if S_plus >= S_minus:
        normal = normal
    else:
        normal = -normal

    S_plus = (diffs @ x_vec >= 0).sum()
    S_minus = diffs.shape[0] - S_plus

    if S_plus >= S_minus:
        x_vec = x_vec
    else:
        x_vec = -x_vec

    if return_diffs and return_mask:
        return np.array([x_vec, np.cross(x_vec, normal), normal]), diffs, mask
    elif return_diffs:
        return np.array([x_vec, np.cross(x_vec, normal), normal]), diffs
    elif return_mask:
        return np.array([x_vec, np.cross(x_vec, normal), normal]), mask
    else:
        return np.array([x_vec, np.cross(x_vec, normal), normal])

In [ ]:
def compute_shot(feature_index, verts, R, mapping_df, axes):
    A, diffs, mask = compute_disambiguated_axes(feature_index, verts, R)
    diffs_rel_coords = diffs @ A.T

    # elevation
    ele = diffs_rel_coords[:,2] >= 0

    # radial
    rad = np.linalg.norm(diffs_rel_coords, axis = 1) <= R / 2

    # azimuth
    num_azimuth = 8
    increment = 2 * np.pi / num_azimuth

    angs = np.atan2(diffs_rel_coords[:,1], diffs_rel_coords[:,0])
    azi = np.round(angs / increment).astype(int)
    azi = azi + 8 * (azi == -4)

    cosines= axes[["zx", "zy", "zz"]][mask].to_numpy() @ axes[["zx", "zy", "zz"]].loc[feature_index].to_numpy()
    cosines = np.round(cosines / 0.2) * 0.2
    cosines = (cosines * 10).astype(int)

    hists = pd.DataFrame({
        "ele": ele,
        "rad": rad,
        "azi": azi,
        "cos": cosines,
        "count": np.zeros(cosines.shape[0])
    }).groupby(["ele", "rad", "azi", "cos"]).count().reset_index()

    this_shot = mapping_df.merge(hists, how = "left", on = ["ele", "rad", "azi", "cos"]).fillna(0)["count"].to_numpy()
    this_shot = this_shot / this_shot.sum()
    return this_shot

In [ ]:
def compute_normal_approx(feature_index, verts, R):
    # Convert to NumPy once
    V = verts
    p0 = V[feature_index]

    # Vectorized distances
    diffs = V - p0
    dists = np.linalg.norm(diffs, axis=1)

    # Radius mask
    mask = dists < R
    diffs = diffs[mask]
    dists = dists[mask]

    # Weights
    w = (R - dists)
    w_sum = np.sum(w)

    if w_sum == 0:
        raise ValueError("No neighbors inside radius")

    # Weighted covariance matrix
    # Equivalent to sum_i w_i * diff_i diff_i^T
    M = (diffs.T * w) @ diffs / w_sum

    # Symmetric eigendecomposition
    eigvals, eigvecs = np.linalg.eigh(M)

    # Smallest eigenvalue → normal direction
    normal = eigvecs[:, 0]
    x_vec = eigvecs[:, 2]

    S_plus = (diffs @ normal >= 0).sum()
    S_minus = diffs.shape[0] - S_plus

    if S_plus >= S_minus:
        normal = normal
    else:
        normal = -normal

    S_plus = (diffs @ x_vec >= 0).sum()
    S_minus = diffs.shape[0] - S_plus

    if S_plus >= S_minus:
        x_vec = x_vec
    else:
        x_vec = -x_vec

    return x_vec, np.cross(x_vec, normal), normal


In [ ]:
def compute_point_axes_non_disambiguated(verts, R):
    rows = []
    for i in range(verts.shape[0]):
        rows.append(np.array(compute_normal_approx(i, verts, R)).reshape(-1))
    out = pd.DataFrame(rows, columns = ["xx", "xy", "xz", "yx", "yy", "yz", "zx", "zy", "zz"])
    return out

In [ ]:
rows = []
for a in [True, False]:
    for b in [True, False]:
        for c in range(-3, 5):
            for d in range(-10, 11, 2):
                rows.append([a,b,c,d])
mapping_df = pd.DataFrame(rows, columns=["ele", "rad", "azi", "cos"])

In [ ]:
p = "datasets/action_smplx_models/male_run.npz"
N = 10000
points1, faces1 = utils.sampled_verts_from_path(p, idx=0, n_points=N, return_faces=True)
points2, faces2 = utils.sampled_verts_from_path(p, idx=10, n_points=N, return_faces=True)

In [ ]:
# R = 1

# axes1 = compute_point_axes_non_disambiguated(points1, R)
# axes2 = compute_point_axes_non_disambiguated(points2, R)

# rows1 = []
# for i in range(N):
#     rows1.append(compute_shot(i, points1, R, mapping_df, axes1))
# rows1 = np.array(rows1)

# rows2 = []
# for i in range(N):
#     rows2.append(compute_shot(i, points2, R, mapping_df, axes2))
# rows2 = np.array(rows2)

In [ ]:
# a = np.ones(N) / N
# b = np.ones(N) / N
# accs = []
# for alpha in [0, .5, 1]:
#     shot_dist = ot.dist(rows1, rows2)
#     euc_dist = ot.dist(points1, points2)
#     M = alpha * shot_dist / shot_dist.mean() + (1 - alpha) * euc_dist / euc_dist.mean()
#     G = ot.solve(M, a, b).plan
#     accs.append(utils.region_accuracy_adjusted(G, faces1, faces2))

In [ ]:
# accs

In [ ]:
# plt.plot(np.arange(0, 1.01, 0.05), accs)

In [ ]:
def calc_mesh_res(faces, verts):
    edge_sum = 0
    edge_count = 0
    for v1i, v2i, v3i in faces:
        edge_sum += np.linalg.norm(verts[v1i] - verts[v2i])
        edge_sum += np.linalg.norm(verts[v1i] - verts[v3i])
        edge_sum += np.linalg.norm(verts[v2i] - verts[v3i])
        edge_count += 3
    return edge_sum / edge_count

In [ ]:
mesh = utils.mesh_from_path(p, idx = 0)
calc_mesh_res(mesh.faces, mesh.vertices)

In [ ]:
import numpy as np
from scipy.spatial import cKDTree

def knn_graph(points, k=3):
    """
    Construct a k-NN graph from a 3D point cloud.

    Parameters
    ----------
    points : (N, 3) numpy array
        3D point cloud
    k : int
        Number of nearest neighbors per point

    Returns
    -------
    edges : list of tuple
        List of undirected edges (i, j)
    """
    points = np.asarray(points)
    N = points.shape[0]

    tree = cKDTree(points)
    _, neighbors = tree.query(points, k=k+1)  # +1 to include the point itself

    edges = set()
    for i in range(N):
        for j in neighbors[i][1:]:  # skip self
            edges.add(tuple(sorted((i, j))))

    return list(edges)


In [ ]:
import numpy as np
import plotly.graph_objects as go

def plot_point_cloud_graph(points, edges):
    """
    Plot a 3D point cloud and its graph connections.

    Parameters
    ----------
    points : (N, 3) numpy array
        3D point cloud
    edges : list of tuple
        List of edges (i, j) indexing into points
    """
    points = np.asarray(points)

    # Scatter plot of points
    scatter = go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        mode='markers',
        marker=dict(size=4),
        name='Points'
    )

    # Build line segments for edges
    x_lines, y_lines, z_lines = [], [], []
    for i, j in edges:
        x_lines += [points[i, 0], points[j, 0], None]
        y_lines += [points[i, 1], points[j, 1], None]
        z_lines += [points[i, 2], points[j, 2], None]

    lines = go.Scatter3d(
        x=x_lines,
        y=y_lines,
        z=z_lines,
        mode='lines',
        line=dict(width=2),
        name='Edges'
    )

    fig = go.Figure(data=[scatter, lines])
    fig.update_layout(
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data'
        ),
        title='3D Point Cloud with k-NN Graph'
    )

    fig.update_layout(scene = dict(aspectmode = "data"))
    fig.update_layout(
        width=600,
        height=1000,   # taller than wide works better for humans
    )
    fig.update_layout(
        scene_camera=dict(
            eye=dict(x=2.5, y=2.5, z=2.5)
        )
    )
    fig.show()


In [ ]:
combined_points = np.append(points1[:1000], points2[:1000], axis = 0)

In [ ]:
plot_point_cloud_graph(combined_points, knn_graph(combined_points))

In [ ]:
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import dijkstra
import utils

def graph_distance_between_clouds(P, Q, k=3):
    """
    Combine two point clouds and compute graph distances.

    Parameters
    ----------
    P, Q : (N, 3) numpy arrays
        Two point clouds of equal size
    k : int
        Number of nearest neighbors for graph construction

    Returns
    -------
    D : (N, N) numpy array
        Graph distance matrix between points in P,
        where distances are shortest-path distances
        in the combined graph weighted by Euclidean edge lengths
    """
    P = np.asarray(P)
    Q = np.asarray(Q)
    assert P.shape == Q.shape

    N = P.shape[0]

    # Combine point clouds: (2N, 3)
    X = np.vstack([P, Q])

    # Build k-NN graph
    tree = cKDTree(X)
    dists, nbrs = tree.query(X, k=k+1)

    rows = []
    cols = []
    weights = []

    for i in range(2 * N):
        for j, dist in zip(nbrs[i][1:], dists[i][1:]):  # skip self
            rows.append(i)
            cols.append(j)
            weights.append(dist)

            # make graph undirected
            rows.append(j)
            cols.append(i)
            weights.append(dist)

    # Sparse weighted adjacency matrix
    A = coo_matrix((weights, (rows, cols)), shape=(2 * N, 2 * N))

    # All-pairs shortest paths from points in P
    dist_full = dijkstra(csgraph=A, directed=False, indices=np.arange(N))

    # Restrict to P → P distances
    D = dist_full[:, :N]

    return D


In [ ]:
p = "datasets/action_smplx_models/male_run.npz"
N = 1000
points1, faces1 = utils.sampled_verts_from_path(p, idx=0, n_points=N, return_faces=True)
points2, faces2 = utils.sampled_verts_from_path(p, idx=10, n_points=N, return_faces=True)
combined_points = np.append(points1, points2, axis = 0)

In [ ]:
plot_point_cloud_graph(combined_points, knn_graph(combined_points, k=3))

In [ ]:
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import dijkstra, connected_components

def graph_distance_between_clouds_connected(P, Q, k_start=3, k_max=None):
    """
    Combine two point clouds and compute graph distances,
    increasing k until the graph is fully connected.

    Parameters
    ----------
    P, Q : (N, 3) numpy arrays
        Two point clouds of equal size
    k_start : int
        Initial number of nearest neighbors
    k_max : int or None
        Maximum k to try (defaults to 2N-1)

    Returns
    -------
    D : (N, N) numpy array
        Graph distance matrix between points in P
    k_used : int
        Final k that achieved connectivity
    """
    P = np.asarray(P)
    Q = np.asarray(Q)
    assert P.shape == Q.shape

    N = P.shape[0]
    X = np.vstack([P, Q])
    M = 2 * N

    if k_max is None:
        k_max = M - 1

    tree = cKDTree(X)

    for k in range(k_start, k_max + 1):
        print("Trying", k)
        dists, nbrs = tree.query(X, k=k + 1)

        rows, cols, weights = [], [], []

        for i in range(M):
            for j, dist in zip(nbrs[i][1:], dists[i][1:]):  # skip self
                rows.append(i)
                cols.append(j)
                weights.append(dist)

                # undirected
                rows.append(j)
                cols.append(i)
                weights.append(dist)

        A = coo_matrix((weights, (rows, cols)), shape=(M, M))

        n_components, _ = connected_components(A, directed=False)

        if n_components == 1:
            break
    else:
        raise RuntimeError("Graph did not become connected up to k_max")

    # Shortest paths from points in P
    dist_full = dijkstra(csgraph=A, directed=False, indices=np.arange(N))

    # Restrict to P → P distances
    D = dist_full[:, N:]

    return D, k


In [ ]:
import ot
import matplotlib.pyplot as plt

In [ ]:
p = "datasets/action_smplx_models/male_run.npz"
N = 1000
points1, faces1 = utils.sampled_verts_from_path(p, idx=0, n_points=N, return_faces=True)
points2, faces2 = utils.sampled_verts_from_path(p, idx=10, n_points=N, return_faces=True)
print("points loaded")
combined_points = np.append(points1, points2, axis = 0)
#D = graph_distance_between_clouds_connected(points1, points2)[0]
D = graph_distance_between_clouds_connected(points1 - points1.mean(axis = 0), points2 - points2.mean(axis = 0))[0]
print("graph distance calculated")
a = np.ones(N) / N
b = np.ones(N) / N
M = ot.dist(points1, points2)
print("euclidean distance calculated")
alphas = np.arange(0, 1.01, 0.1)
region_accs = []
av_reg_dists = []
for alpha in alphas:
    interp_dist = alpha * M / M.mean() + (1 - alpha) * D / D.mean()
    this_G = ot.solve(interp_dist, a, b).plan
    region_accs.append(utils.region_accuracy(this_G, faces1, faces2))
    av_reg_dists.append(utils.average_region_distance(this_G, faces1,faces2))
    print("alpha", alpha, "done")
plt.plot(alphas, region_accs, label = "Region Accuracy")
plt.plot(alphas, av_reg_dists, label = "Average Region Distance")
plt.legend()
plt.ylim(0,1)

In [ ]:
region_accs

In [ ]:
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
import plotly.express as px

def plot_connected_components(points, k=3):
    """
    Plot a 3D point cloud with connected components highlighted.

    Parameters
    ----------
    points : (N, 3) numpy array
        3D point cloud
    k : int
        Number of nearest neighbors (default: 3)
    """
    points = np.asarray(points)
    N = points.shape[0]

    # Build k-NN graph
    tree = cKDTree(points)
    _, nbrs = tree.query(points, k=k + 1)

    rows, cols = [], []
    for i in range(N):
        for j in nbrs[i][1:]:  # skip self
            rows.append(i)
            cols.append(j)
            rows.append(j)   # undirected
            cols.append(i)

    A = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(N, N))

    # Connected components
    n_components, labels = connected_components(A, directed=False)

    # Plot
    fig = px.scatter_3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        color=labels.astype(str),
        title=f"Connected Components (k = {k}, components = {n_components})"
    )

    fig.update_traces(marker=dict(size=4))
    fig.update_layout(scene=dict(aspectmode='data'))

    fig.show()


In [ ]:
combined_points = np.append(points1[:10000], points2[:10000], axis = 0)

In [ ]:
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

def connect_components_minimally(points, k=3):
    """
    Build a k=3 NN graph and minimally connect components
    by adding shortest inter-component edges until connected.

    Parameters
    ----------
    points : (N, 3) numpy array
        3D point cloud
    k : int
        Number of nearest neighbors

    Returns
    -------
    edges : set of tuple
        Set of undirected edges (i, j)
    """
    points = np.asarray(points)
    N = points.shape[0]

    # Initial kNN graph
    tree = cKDTree(points)
    _, nbrs = tree.query(points, k=k + 1)

    edges = set()
    for i in range(N):
        for j in nbrs[i][1:]:
            edges.add(tuple(sorted((i, j))))

    while True:
        # Build adjacency
        rows, cols = zip(*[(i, j) for i, j in edges] +
                         [(j, i) for i, j in edges])
        A = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(N, N))

        n_components, labels = connected_components(A, directed=False)

        if n_components == 1:
            break

        # Pick one component
        c0 = 0
        idx_c0 = np.where(labels == c0)[0]
        idx_rest = np.where(labels != c0)[0]

        # Find closest pair (brute force, but clear)
        P = points[idx_c0]
        Q = points[idx_rest]

        dists = np.linalg.norm(P[:, None, :] - Q[None, :, :], axis=2)
        i_min, j_min = np.unravel_index(np.argmin(dists), dists.shape)

        u = idx_c0[i_min]
        v = idx_rest[j_min]

        edges.add(tuple(sorted((u, v))))

    return edges


In [ ]:
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components, dijkstra

def graph_distance_between_clouds_minimally_connected(P, Q, k=3):
    """
    Construct a minimally connected kNN graph on two point clouds
    and compute graph distances.

    Parameters
    ----------
    P, Q : (N, 3) numpy arrays
        Two point clouds of equal size
    k : int
        Number of nearest neighbors (default: 3)

    Returns
    -------
    D : (N, N) numpy array
        Graph distance matrix from P to Q
    """
    P = np.asarray(P)
    Q = np.asarray(Q)
    assert P.shape == Q.shape

    N = P.shape[0]
    X = np.vstack([P, Q])          # (2N, 3)
    M = 2 * N

    # --- Step 1: initial kNN graph ---
    tree = cKDTree(X)
    dists, nbrs = tree.query(X, k=k + 1)

    edges = {}  # (i, j) -> weight
    for i in range(M):
        for j, dist in zip(nbrs[i][1:], dists[i][1:]):
            a, b = sorted((i, j))
            edges[(a, b)] = dist

    # --- Step 2: minimally connect components ---
    while True:
        rows, cols, weights = [], [], []
        for (i, j), w in edges.items():
            rows += [i, j]
            cols += [j, i]
            weights += [w, w]

        A = coo_matrix((weights, (rows, cols)), shape=(M, M))
        n_components, labels = connected_components(A, directed=False)

        if n_components == 1:
            break

        # pick one component
        c0 = labels[0]
        idx_c0 = np.where(labels == c0)[0]
        idx_rest = np.where(labels != c0)[0]

        # find closest inter-component pair
        P0 = X[idx_c0]
        P1 = X[idx_rest]

        dmat = np.linalg.norm(P0[:, None, :] - P1[None, :, :], axis=2)
        i_min, j_min = np.unravel_index(np.argmin(dmat), dmat.shape)

        u = idx_c0[i_min]
        v = idx_rest[j_min]
        w = dmat[i_min, j_min]

        a, b = sorted((u, v))
        edges[(a, b)] = w

    # --- Step 3: shortest-path distances ---
    rows, cols, weights = [], [], []
    for (i, j), w in edges.items():
        rows += [i, j]
        cols += [j, i]
        weights += [w, w]

    A = coo_matrix((weights, (rows, cols)), shape=(M, M))

    # distances from P nodes (0..N-1) to all nodes
    dist_full = dijkstra(A, directed=False, indices=np.arange(N))

    # extract P -> Q distances
    D = dist_full[:, N:]

    return D


In [ ]:
p = "datasets/action_smplx_models/male_run.npz"
N = 1000
points1, faces1 = utils.sampled_verts_from_path(p, idx=0, n_points=N, return_faces=True)
points2, faces2 = utils.sampled_verts_from_path(p, idx=10, n_points=N, return_faces=True)
print("points loaded")
combined_points = np.append(points1, points2, axis = 0)
#D = graph_distance_between_clouds_connected(points1, points2)[0]
D = graph_distance_between_clouds_minimally_connected(points1 - points1.mean(axis = 0), points2 - points2.mean(axis = 0), k = 3)
print("graph distance calculated")
a = np.ones(N) / N
b = np.ones(N) / N
M = ot.dist(points1, points2)
print("euclidean distance calculated")
alphas = np.arange(0, 1.01, 0.05)
region_accs = []
av_reg_dists = []
for alpha in alphas:
    interp_dist = alpha * M / M.mean() + (1 - alpha) * D / D.mean()
    this_G = ot.solve(interp_dist, a, b).plan
    region_accs.append(utils.region_accuracy(this_G, faces1, faces2))
    av_reg_dists.append(utils.average_region_distance(this_G, faces1,faces2))
    print("alpha", alpha, "done")
plt.plot(alphas, region_accs, label = "Region Accuracy")
plt.plot(alphas, av_reg_dists, label = "Average Region Distance")
plt.legend()
plt.ylim(0,1)

In [ ]:
interp_dist

In [ ]:
av_reg_dists

In [ ]:
D = graph_distance_between_clouds_minimally_connected(points1, points2)

In [ ]:
G = ot.solve(D, np.ones(N) / N, np.ones(N) / N)

In [ ]:
npz = np.load("datasets/action_smplx_models/male_run.npz")

In [ ]:
npz["mocap_time_length"] * 120

In [ ]:
utils.mesh_from_path("datasets/action_smplx_models/male_run.npz", idx = 326)

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize = (20, 10))
ax[0].plot([1, 2, 3], [5, 6, 1], label = "yo")
ax[0].legend()
plt.show()

In [ ]:
import utils
import pandas as pd

In [ ]:
points1, faces1 = utils.sampled_verts_from_random_action(return_faces = True)
df = pd.DataFrame({
    "x": points1[:,0],
    "y": points1[:,1],
    "z": points1[:,2],
    "regions": utils.faces_to_regions(faces1)
})
utils.plot_arr(df.groupby("regions").median().to_numpy())

In [ ]:
df.groupby("regions").median()

In [ ]:
utils.plot_arr(points1)

In [ ]:
import ot
import numpy as np
LABELS = [
    "head",
    "left_foot",
    "left_forearm",
    "left_hand",
    "left_shin",
    "left_thigh",
    "left_upper_arm",
    "lower_torso",
    "pelvis",
    "right_foot",
    "right_forearm",
    "right_hand",
    "right_shin",
    "right_thigh",
    "right_upper_arm",
    "upper_torso",
]

LABEL_TO_INDEX = {label: i for i, label in enumerate(LABELS)}

D = np.array([
    [0, 6, 4, 5, 6, 5, 3, 2, 3, 6, 4, 5, 6, 5, 3, 1],
    [6, 0, 6, 7, 1, 2, 4, 3, 2, 4, 6, 7, 3, 2, 4, 4],
    [4, 6, 0, 1, 6, 5, 1, 4, 5, 6, 4, 5, 6, 5, 3, 2],
    [5, 7, 1, 0, 7, 6, 2, 5, 6, 7, 5, 6, 7, 6, 4, 3],
    [6, 1, 6, 7, 0, 1, 5, 4, 3, 5, 7, 8, 4, 3, 5, 5],
    [5, 2, 5, 6, 1, 0, 4, 3, 2, 4, 6, 7, 3, 2, 4, 4],
    [3, 4, 1, 2, 5, 4, 0, 3, 4, 5, 3, 4, 5, 4, 2, 1],
    [2, 3, 4, 5, 4, 3, 3, 0, 1, 3, 5, 6, 4, 3, 3, 1],
    [3, 2, 5, 6, 3, 2, 4, 1, 0, 2, 6, 7, 3, 2, 4, 2],
    [6, 4, 6, 7, 5, 4, 5, 3, 2, 0, 6, 7, 1, 2, 4, 4],
    [4, 6, 4, 5, 7, 6, 3, 5, 6, 6, 0, 1, 6, 5, 1, 2],
    [5, 7, 5, 6, 8, 7, 4, 6, 7, 7, 1, 0, 7, 6, 2, 3],
    [6, 3, 6, 7, 4, 3, 5, 4, 3, 1, 6, 7, 0, 1, 5, 5],
    [5, 2, 5, 6, 3, 2, 4, 3, 2, 2, 5, 6, 1, 0, 4, 4],
    [3, 4, 3, 4, 5, 4, 2, 3, 4, 4, 1, 2, 5, 4, 0, 1],
    [1, 4, 2, 3, 5, 4, 1, 1, 2, 4, 2, 3, 5, 4, 1, 0],
], dtype=np.int8)

In [ ]:
points1, faces1 = utils.sampled_verts_from_path("datasets/action_smplx_models/male_run.npz", idx = 0, n_points = 100, return_faces=True)
points2, faces2 = utils.sampled_verts_from_path("datasets/action_smplx_models/male_run.npz", idx = 10, n_points = 100, return_faces=True)
regions1 = utils.faces_to_regions(faces1)
regions2 = utils.faces_to_regions(((G / G.max()) @ faces2).astype(int))
M = ot.dist(points1, points2)
a = np.ones(100) / 100
b = np.ones(100) / 100
G = ot.solve(M, a, b).plan

In [ ]:
idx1 = np.fromiter((LABEL_TO_INDEX[r] for r in regions1), dtype=np.int64)
idx2 = np.fromiter((LABEL_TO_INDEX[r] for r in regions2), dtype=np.int64)

In [ ]:
D[idx1, idx2].mean()

In [ ]:
df.groupby("regions").median().to_numpy()

In [ ]:
import utils

In [ ]:
utils.plot_median_skeleton